In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split
from utils import *
import prompt
from sklearn.cluster import KMeans
import openai

# This is not useful. change client in submit_end2end.py
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY_YALE"])
#client = openai.OpenAI()

import re

In [ ]:
representetive_gene_list = (repository_root() / "examples/merfish/representative_genes.txt").read_text().splitlines()


# data

In [ ]:
config = load_config("configs/config_finetunePro_merfish.yaml")
config.data_name = "MERFISH_25"
config.refresh_paths()
name_truth = config.name_truth

In [ ]:
noise_p = 0.1

In [ ]:
# --- Load data ---
data_path = str(dataset_file("merfish", config.data_name))
adata = sc.read_h5ad(data_path) 
# rename the column of cell_class to cell_type
adata.obs.rename(columns={'cell_class': 'cell_type'}, inplace=True)

# clean the cell ID to save token
adata.obs_names = list(range(len(adata)))
adata.obs_names = adata.obs_names.astype(str)

# Remove rows with NaN values in 'layer_guess'
adata = adata[~adata.obs[name_truth].isna()].copy()

# Verify that NaNs have been removed
remaining_nan_count = adata.obs[name_truth].isna().sum()
print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
adata.obs = adata.obs.join(pos_data)

# rename cell types
celltype_rename = {
    'Astrocyte' : 'Astrocyte',
 'Endothelial 1': 'Endothelial',
 'OD Mature 2': 'Mature oligodendrocytes',
 'Inhibitory': 'Inhibitory',
 'OD Immature 1': 'Immature oligodendrocytes',
 'Excitatory': 'Excitatory',
 'Endothelial 3': 'Endothelial',
 'Microglia': 'Microglia',
 'OD Mature 1': 'Mature oligodendrocytes',
 'Pericytes': 'Pericytes',
 'OD Mature 4': 'Mature oligodendrocytes',
 'Endothelial 2': 'Endothelial',
 'OD Mature 3': 'Mature oligodendrocytes',
 'OD Immature 2': 'Immature oligodendrocytes',
 'Ependymal': 'Ependymal'
}
adata.obs['cell_type'] = adata.obs['cell_type'].map(celltype_rename)
celltype_data = adata.obs[['cell_type']]

# add noise
noise_X = adata.X.copy()
random_indices = np.random.choice(noise_X.shape[0], size=int(noise_p * noise_X.shape[0]), replace=False)
noise_X[random_indices] = 0
adata.X = noise_X

sc.pp.filter_genes(adata, min_cells=5)
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.scale(adata)


# --- Compute adjacency matrix ---
r = config.r
adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
# add diagonal to the adj_matrix
adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

# --- Generate one-hot encoded matrix ---
one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
one_hot_matrix = one_hot_df.values
one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(one_hot_matrix)
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
print(f"Mean of n_neighbors: {np.mean(n_neighbors)}")
# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized.toarray(), 
                              index=celltype_data.index, 
                              columns=one_hot_df.columns.str.lstrip('_'))



In [ ]:
top_genes = representetive_gene_list

# --- Calculate neighbor genes ---
neighbor_genes = adj_matrix.dot(adata[:,top_genes].X)

# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized_genes = neighbor_genes / n_neighbors_col

neighbor_normalized_df_genes = pd.DataFrame(neighbor_matrix_normalized_genes, 
                              index=adata.obs_names, 
                              columns=top_genes)



# check noise
randomly set different proportion of value to 0, check the list of genes

In [ ]:
# Get top 10 genes for each cell (row)
def get_top_genes(row):
    # Sort values in descending order and get top 10 gene names
    return pd.Series(row.nlargest(10).index)

# Calculate which cells have different top genes
def compare_top_genes(original_df, noise_df):
    # Check if the sets of top genes are different for each cell
    cells_changed = (original_df != noise_df).any(axis=1)
    
    # Calculate percentage
    percent_changed = (cells_changed.sum() / len(cells_changed)) * 100
    
    # Get number of cells changed
    num_cells_changed = cells_changed.sum()
    
    return percent_changed, num_cells_changed

def calculate_gene_set_differences(original_df, noise_df):
    # Convert rows to sets and calculate difference sizes
    def get_intersection_size(row1, row2):
        set1 = set(row1)
        set2 = set(row2)
        return len(set1.intersection(set2))
    
    # Calculate differences for each cell
    intersection_size = pd.DataFrame(index=original_df.index)
    intersection_size['num_same_genes'] = [
        get_intersection_size(original_df.loc[idx], noise_df.loc[idx])
        for idx in original_df.index
    ]
    num_genes = len(original_df.columns)
    # Calculate summary statistics
    summary_stats = {
        'total_cells': len(intersection_size),
        'cells_with_differences': (intersection_size['num_same_genes'] < num_genes).sum(),
        'mean_differences': num_genes-intersection_size['num_same_genes'].mean(),
        'median_differences': num_genes-intersection_size['num_same_genes'].median(),
        'max_differences': num_genes-intersection_size['num_same_genes'].min(),
        'distribution': intersection_size['num_same_genes'].value_counts().sort_index()
    }
    
    return intersection_size, summary_stats



# Apply the function to each row and create a new dataframe with the results
original_top_genes_df = neighbor_normalized_df_genes.apply(get_top_genes, axis=1)

# Rename columns to be more descriptive
original_top_genes_df.columns = [f'Gene_{i+1}' for i in range(10)]

In [ ]:
# original data

adata = sc.read_h5ad(data_path) 
# rename the column of cell_class to cell_type
adata.obs.rename(columns={'cell_class': 'cell_type'}, inplace=True)

# clean the cell ID to save token
adata.obs_names = list(range(len(adata)))
adata.obs_names = adata.obs_names.astype(str)

# Remove rows with NaN values in 'layer_guess'
adata = adata[~adata.obs[name_truth].isna()].copy()

# Verify that NaNs have been removed
remaining_nan_count = adata.obs[name_truth].isna().sum()
print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
adata.obs = adata.obs.join(pos_data)

# rename cell types
celltype_rename = {
    'Astrocyte' : 'Astrocyte',
 'Endothelial 1': 'Endothelial',
 'OD Mature 2': 'Mature oligodendrocytes',
 'Inhibitory': 'Inhibitory',
 'OD Immature 1': 'Immature oligodendrocytes',
 'Excitatory': 'Excitatory',
 'Endothelial 3': 'Endothelial',
 'Microglia': 'Microglia',
 'OD Mature 1': 'Mature oligodendrocytes',
 'Pericytes': 'Pericytes',
 'OD Mature 4': 'Mature oligodendrocytes',
 'Endothelial 2': 'Endothelial',
 'OD Mature 3': 'Mature oligodendrocytes',
 'OD Immature 2': 'Immature oligodendrocytes',
 'Ependymal': 'Ependymal'
}
adata.obs['cell_type'] = adata.obs['cell_type'].map(celltype_rename)
celltype_data = adata.obs[['cell_type']]





In [ ]:
noise_p = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
result_df = pd.DataFrame()
for p in noise_p:
    noise_X = adata.X.copy()
    random_indices = np.random.choice(noise_X.shape[0], size=int(p * noise_X.shape[0]), replace=False)
    noise_X[random_indices] = 0
    noise_adata = adata.copy()
    noise_adata.X = noise_X
    sc.pp.filter_genes(noise_adata, min_cells=5)
    sc.pp.normalize_total(noise_adata, inplace=True)
    sc.pp.log1p(noise_adata)
    sc.pp.scale(noise_adata)
    # --- Calculate neighbor genes ---
    noise_neighbor_genes = adj_matrix.dot(noise_adata[:,top_genes].X)

    # Perform element-wise division between neighbor_count and n_neighbors_col
    noise_neighbor_matrix_normalized_genes = noise_neighbor_genes / n_neighbors_col

    noise_neighbor_normalized_df_genes = pd.DataFrame(noise_neighbor_matrix_normalized_genes, 
                                index=noise_adata.obs_names, 
                                columns=top_genes)
    # Apply the function to each row and create a new dataframe with the results
    noise_top_genes_df = noise_neighbor_normalized_df_genes.apply(get_top_genes, axis=1)
    # Rename columns to be more descriptive
    noise_top_genes_df.columns = [f'Gene_{i+1}' for i in range(10)]
    
    # Calculate differences
    intersection_size, stats = calculate_gene_set_differences(original_top_genes_df, noise_top_genes_df)

    # # Print summary statistics
    # print(f"Total number of cells: {stats['total_cells']}")
    # print(f"Number of cells with different gene sets: {stats['cells_with_differences']}")
    # print(f"Mean number of different genes per cell: {stats['mean_differences']:.2f}")
    # print(f"Median number of different genes per cell: {stats['median_differences']:.2f}")
    # print(f"Maximum number of different genes in a cell: {stats['max_differences']}")
    
    # Add results to DataFrame
    stats_df = pd.DataFrame({
        'noise_level': [p],
        'total_cells': [stats['total_cells']],
        'cells_with_differences': [stats['cells_with_differences']], 
        'mean_differences': [stats['mean_differences']],
        'median_differences': [stats['median_differences']],
        'max_differences': [stats['max_differences']]
    })
    
    result_df = pd.concat([result_df, stats_df], ignore_index=True)


   


In [ ]:
result_df



# sample data

In [ ]:
# 设定随机种子
seed = 42  # 你可以根据需要修改这个值

# 定义分割比例 p (比如 0.7 表示 70% 数据用于训练，30% 数据用于测试)
p = config.prototype_p

# 分割数据集为训练集和测试集
train_neighbor_normalized_df, val_neighbor_normalized_df = train_test_split(neighbor_normalized_df, 
                                                            test_size=1-p, 
                                                            random_state=seed,
                                                            stratify=adata.obs[config.name_truth]
                                                           )


train_neighbor_normalized_df_genes, val_neighbor_normalized_df_genes = train_test_split(neighbor_normalized_df_genes, 
                                                            test_size=1-p, 
                                                            random_state=seed,
                                                            stratify=adata.obs[config.name_truth]
                                                           )


In [ ]:
# check sample distribution
adata.obs[config.name_truth].loc[train_neighbor_normalized_df.index].value_counts()

In [ ]:
# --- prototype ---
# IMPORTANT: prototype is calculated on the train data
# calculate prototype
train_neighbor_df = train_neighbor_normalized_df.join(train_neighbor_normalized_df_genes).copy()
one_shot_df = pd.concat([adata.obs[config.name_truth].loc[train_neighbor_df.index], train_neighbor_df], axis=1).groupby(config.name_truth, observed=False).mean()
print(one_shot_df.index)


In [ ]:
# validation in finetune is not necessary
# # finetune train and val data
# _, val_for_finetune = train_test_split(val_neighbor_normalized_df, 
#                                                             test_size=0.1, 
#                                                             random_state=seed
#                                                            )

# adata.obs.loc[val_for_finetune.index, config.name_truth].value_counts()

# prompt

In [ ]:
unique_layers = adata.obs[name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

# the full cell type names are already in the data
unique_celltypes = adata.obs['cell_type'].unique()
cell_names_mapping = {celltype: celltype for _, celltype in enumerate(unique_celltypes)}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping

config.cell_names = neighbor_normalized_df.columns
config.gene_names = top_genes


# generate Comparison-based Prompt
config.system_prompt = prompt.CP_celltype_geneorder(one_shot_df, config)
if config.with_negatives:
    # get top 3 genes for each row in one_shot_df
    neg_top_3_genes = one_shot_df[config.gene_names].apply(lambda x: x.nlargest(3).index.tolist(), axis=1)
    config.neg_top_3_genes = neg_top_3_genes


In [ ]:
print(prompt.CP_celltype_geneorder(one_shot_df, config))


In [ ]:
print(prompt.finetune_user_celltype_geneorder(train_neighbor_normalized_df, train_neighbor_normalized_df_genes, 7, config))
print(prompt.finetune_assistant(train_neighbor_normalized_df, 7, adata.obs[config.name_truth]))


# GPT-4o-mini

In [ ]:
print(f"finetune_json/{config.data_name}_{config.model_type}/")

In [ ]:
# generate json for finetune
output_folder = f"finetune_json/{config.data_name}_{config.model_type}/"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

train_output_file = f"{output_folder}{config.data_name}_train_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
print(f"Generating json for training into {train_output_file}")
with open(train_output_file, 'w') as f:
    for i in range(train_neighbor_normalized_df.shape[0]):
        system_p = config.system_prompt
        user_p = prompt.finetune_user_celltype_geneorder(train_neighbor_normalized_df, train_neighbor_normalized_df_genes, i, config)
        assistant_p = prompt.finetune_assistant(train_neighbor_normalized_df, i, adata.obs[config.name_truth])
        
        row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
        json_str = json.dumps(row_data)
        f.write(json_str + '\n')  

# val_output_file = f"{output_folder}{config.data_name}_val_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
# print(f"Generating json for validation into {val_output_file}")
# with open(val_output_file, 'w') as f:
#     for i in range(val_for_finetune.shape[0]):
#         system_p = config.system_prompt
#         user_p = prompt.finetune_user_deconv(val_for_finetune, i, config)
#         assistant_p = prompt.finetune_assistant(val_for_finetune, i, adata.obs[config.name_truth])

#         row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
#         json_str = json.dumps(row_data)
#         f.write(json_str + '\n')  

In [ ]:
print(system_p + user_p + "\n" + assistant_p)


In [ ]:
train_file = client.files.create(
  file=open(train_output_file, "rb"),
  purpose="fine-tune"
)

# val_file = client.files.create(
#   file=open(val_output_file, "rb"),
#   purpose="fine-tune"
# )

finetune_job = client.fine_tuning.jobs.create(
  training_file=train_file.id,
  # validation_file=val_file.id,
  model="gpt-4o-mini-2024-07-18",
  suffix=f"{config.model_type}_{config.r}_withnegatives"  # default is personal
)


# gemini 1.5 flash

In [ ]:
import google

In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.environ["API_KEY"])
for model_info in genai.list_tuned_models():
    print(model_info.name)

In [ ]:

tunable_models = [
    m for m in genai.list_models()
    if "createTunedModel" in m.supported_generation_methods]
tunable_models

In [ ]:
base_model = "models/gemini-1.5-flash-001-tuning"
training_data = []
for i in range(train_neighbor_normalized_df.shape[0]):
    system_p = prompt.finetune_system_deconv(config)
    user_p = prompt.finetune_user_deconv(train_neighbor_normalized_df, i, config)
    assistant_p = prompt.finetune_assistant(train_neighbor_normalized_df, i, adata.obs[config.name_truth])
    training_data.append({'text_input': system_p + user_p, 'output': assistant_p})
    



In [ ]:
training_data[:4]

In [ ]:
operation = genai.create_tuned_model(
    # You can use a tuned model here too. Set `source_model="tunedModels/..."`
    display_name=config.data_name,
    source_model=base_model,
    epoch_count=30,
    batch_size=4,
    learning_rate=0.001,
    training_data=training_data,
)

In [ ]:
for status in operation.wait_bar():
    time.sleep(10)

In [ ]:
result = operation.result()

In [ ]:
import seaborn as sns
model = operation.result()  # model = genai.get_tuned_model()
snapshots = pd.DataFrame(model.tuning_task.snapshots)

sns.lineplot(data=snapshots, x = 'epoch', y='mean_loss')

# Use the finetuned model

## generate json

In [ ]:
# add model name to config
# config.gpt_model = "ft:gpt-4o-mini-2024-07-18:whh:finetune-merfish270-3:ARMu34Br"

# add model name to config and reload config
config = load_config("configs/config_finetunePro_merfish.yaml")
config.data_name = "MERFISH_25"
config.refresh_paths()
name_truth = config.name_truth

unique_layers = adata.obs[name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

# the full cell type names are already in the data
unique_celltypes = adata.obs['cell_type'].unique()
cell_names_mapping = {celltype: celltype for _, celltype in enumerate(unique_celltypes)}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping

config.cell_names = neighbor_normalized_df.columns
config.gene_names = top_genes

config.system_prompt = prompt.CP_celltype_geneorder(one_shot_df, config)
if config.with_negatives:
    # get top 3 genes for each row in one_shot_df
    neg_top_3_genes = one_shot_df[config.gene_names].apply(lambda x: x.nlargest(3).index.tolist(), axis=1)
    config.neg_top_3_genes = neg_top_3_genes



In [ ]:
print(config.folder_path)

In [ ]:
print(config.gpt_model)

In [ ]:
config.replicate = "_rep1Noise01"



In [ ]:
# make sure the gpt_model is correct
generate_json_end2end(val_neighbor_normalized_df, config, prompt_func=prompt.finetune_user_celltype_geneorder, n_rows=1, batch_size=2000, df_extra=val_neighbor_normalized_df_genes)

## submit

In [ ]:

import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_finetunePro_merfish.yaml {config.data_name} {config.replicate} > outs/{config.data_name}_finetunePro_25{config.replicate}_{config.prototype_p}.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
if len(val_neighbor_normalized_df) > 2000:
    n_batch = 2  
else:
    n_batch = 1

for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['finetunePro_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)
gpt_results_df.index = gpt_results_df.index.str.replace("id_", "")

In [ ]:


# Get the list of keywords from domain_mapping
keywords = list(domain_mapping.values())

# Create a regex pattern to capture any keyword possibly surrounded by other text
pattern = r'.*(' + '|'.join(map(re.escape, keywords)) + r')[\*\.\s]*.*'

# Replace the entire string with the captured keyword only if it could be not unknown
for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<3].index:
    gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"].str.replace(pattern, r'\1', regex=True)


In [ ]:
gpt_results_df.value_counts()

In [ ]:
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "mpa", "finetunePro_gpt4o_mini"] = "MPA"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "MpA", "finetunePro_gpt4o_mini"] = "MPA"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "FX", "finetunePro_gpt4o_mini"] = "fx"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == " outcomes: 'MPN'", "finetunePro_gpt4o_mini"] = "MPN"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "pv", "finetunePro_gpt4o_mini"] = "PV"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == " PV", "finetunePro_gpt4o_mini"] = "PV"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == " fx", "finetunePro_gpt4o_mini"] = "fx"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "Outputs: 'MPN'", "finetunePro_gpt4o_mini"] = "MPN"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "PVH rez", "finetunePro_gpt4o_mini"] = "PVH"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == " BST", "finetunePro_gpt4o_mini"] = "BST"

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
# if the number of the cell type is less than 4, set it to unknown
for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<4].index:
    gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = "unknown"

In [ ]:
# # manually retrieve batch output
# output_file_name = f"{output_path}/response_BZ5_1_{use_full_name}_{with_self_type}_{with_region_name}_{Graph_type}_{with_negatives}_{with_CoT}_{with_count_numbers}.txt"
# batch_id = "batch_66f62020d11c81909424c1423da11a70"
# file_response = client.files.content(client.batches.retrieve(batch_id).output_file_id)
# # Open the file in write mode and save the string
# with open(output_file_name, 'w') as file:
#     file.write(file_response.text)

## plot and save

In [ ]:
val_adata = adata[val_neighbor_normalized_df.index].copy()
val_adata.obs = val_adata.obs.join(gpt_results_df)
val_adata.obs['finetunePro_gpt4o_mini'] = val_adata.obs['finetunePro_gpt4o_mini'].fillna("unknown")
sc.pl.scatter(val_adata, x="x", y="y", color="finetunePro_gpt4o_mini", title =  f"finetunePro_gpt4o_mini")

print(adjusted_rand_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini']))

In [ ]:
print(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")



In [ ]:
print(f"NMI: {normalized_mutual_info_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini'])}")


## refine the niche

In [ ]:
config.r_factor = 1
val_adj_matrix, _ = sparse_adjacency(pos_data.loc[val_neighbor_normalized_df.index], threshold=r*config.r_factor)
refined_niche = relabel_cells(val_adj_matrix.toarray(), val_adata.obs['finetunePro_gpt4o_mini'])

val_adata.obs['finetunePro_gpt4o_mini_refined'] = refined_niche
sc.pl.scatter(val_adata, x="x", y="y", color="finetunePro_gpt4o_mini_refined", title =  f"finetunePro_gpt4o_mini_refined")

print(adjusted_rand_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini_refined']))

In [ ]:
val_adata.obs.to_csv(f"./gpt4omini_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
print(f"NMI: {normalized_mutual_info_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini_refined'])}")


## gather replicate results

In [ ]:
save_folder = "./test_p_results"

In [ ]:
if len(val_neighbor_normalized_df) > 3000:
    n_batch = 2  
else:
    n_batch = 1
config.r_factor = 1
val_adj_matrix, _ = sparse_adjacency(pos_data.loc[val_neighbor_normalized_df.index], threshold=r*config.r_factor)

for replicate in ['_rep1R100', '_rep2R100', '_rep3R100', '_rep4R100', '_rep5R100']:
    config.replicate = replicate
    # 初始化空列表以保存custom_id和content
    gpt_results_df = pd.DataFrame()
    for i in range(1,n_batch+1):
        save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
        output_file_name = f"{config.output_path}/{save_name}"
        # 打开文件并逐行读取
        with open(output_file_name, 'r', encoding='utf-8') as file:
            for line in file:
                try:
                    # 解析每一行的json字符串
                    json_data = json.loads(line.strip())
                    
                    # 提取custom_id和content信息
                    custom_id = json_data['custom_id']
                    content = json_data['response']['body']['choices'][0]['message']['content']
                    content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                    content = content.replace("Layer ", "Layer")
                    # extract outputs
                    extract_dict = extract_output_microenvironments(content)
                    # extract_dict = extract_last_braces(content)

                    # 将提取到的信息添加到数据框中
                    gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                        
                except json.JSONDecodeError:
                    print(f"无法解析JSON字符串: {line}")
    gpt_results_df.columns = ['finetunePro_gpt4o_mini']
    gpt_results_df.index = gpt_results_df.index.astype(str)
    gpt_results_df.index = gpt_results_df.index.str.replace("id_", "")

    # Get the list of keywords from domain_mapping
    keywords = list(domain_mapping.values())

    # Create a regex pattern to capture any keyword possibly surrounded by other text
    pattern = r'.*(' + '|'.join(map(re.escape, keywords)) + r')[\*\.\s]*.*'

    # Replace the entire string with the captured keyword only if it could be not unknown
    for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<3].index:
        gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"].str.replace(pattern, r'\1', regex=True)


    # if the number of the cell type is less than 3, set it to unknown
    for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<3].index:
        gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = "unknown"
    
    if 'unknown' in gpt_results_df.finetunePro_gpt4o_mini.value_counts().index:
        print(f"unknown: {gpt_results_df.value_counts()['unknown']}")

    # merge results to obs
    val_adata = adata[val_neighbor_normalized_df.index].copy()
    val_adata.obs = val_adata.obs.join(gpt_results_df)
    val_adata.obs['finetunePro_gpt4o_mini'] = val_adata.obs['finetunePro_gpt4o_mini'].fillna("unknown")

    # refine the niche
    refined_niche = relabel_cells(val_adj_matrix.toarray(), val_adata.obs['finetunePro_gpt4o_mini'])

    val_adata.obs['finetunePro_gpt4o_mini_refined'] = refined_niche

    val_adata.obs.to_csv(f"{save_folder}/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")



# test

## load test data 
Prototype in prompt is based on training data

In [ ]:
# add model name to config
# config.gpt_model = "ft:gpt-4o-mini-2024-07-18:whh:finetune-merfish270-3:ARMu34Br"

# add model name to config and reload config
config = load_config("configs/config_finetunePro_merfish.yaml")
config.data_name = "MERFISH_29"  # IMPORTANT !!!!
config.refresh_paths()
name_truth = config.name_truth




In [ ]:
# --- Load data ---
data_path = str(dataset_file("merfish", config.data_name))
adata = sc.read_h5ad(data_path) 
# rename the column of cell_class to cell_type
adata.obs.rename(columns={'cell_class': 'cell_type'}, inplace=True)

# clean the cell ID to save token
adata.obs_names = list(range(len(adata)))
adata.obs_names = adata.obs_names.astype(str)

# Remove rows with NaN values in 'layer_guess'
adata = adata[~adata.obs[name_truth].isna()].copy()

# Verify that NaNs have been removed
remaining_nan_count = adata.obs[name_truth].isna().sum()
print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
adata.obs = adata.obs.join(pos_data)

# rename cell types
celltype_rename = {
    'Astrocyte' : 'Astrocyte',
 'Endothelial 1': 'Endothelial',
 'OD Mature 2': 'Mature oligodendrocytes',
 'Inhibitory': 'Inhibitory',
 'OD Immature 1': 'Immature oligodendrocytes',
 'Excitatory': 'Excitatory',
 'Endothelial 3': 'Endothelial',
 'Microglia': 'Microglia',
 'OD Mature 1': 'Mature oligodendrocytes',
 'Pericytes': 'Pericytes',
 'OD Mature 4': 'Mature oligodendrocytes',
 'Endothelial 2': 'Endothelial',
 'OD Mature 3': 'Mature oligodendrocytes',
 'OD Immature 2': 'Immature oligodendrocytes',
 'Ependymal': 'Ependymal'
}
adata.obs['cell_type'] = adata.obs['cell_type'].map(celltype_rename)
celltype_data = adata.obs[['cell_type']]

sc.pp.filter_genes(adata, min_cells=5)
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.scale(adata)


# --- Compute adjacency matrix ---
r = config.r
adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
# add diagonal to the adj_matrix
adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

# --- Generate one-hot encoded matrix ---
one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
one_hot_matrix = one_hot_df.values
one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(one_hot_matrix)
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
print(f"Mean of n_neighbors: {np.mean(n_neighbors)}")
# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized.toarray(), 
                              index=celltype_data.index, 
                              columns=one_hot_df.columns.str.lstrip('_'))



In [ ]:
# use the same top_genes as training data

# --- Calculate neighbor genes ---
neighbor_genes = adj_matrix.dot(adata[:,top_genes].X)

# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized_genes = neighbor_genes / n_neighbors_col

neighbor_normalized_df_genes = pd.DataFrame(neighbor_matrix_normalized_genes, 
                              index=adata.obs_names, 
                              columns=top_genes)


## test GPT

In [ ]:
config.replicate="_rep3R100"

In [ ]:
unique_layers = adata.obs[name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

# the full cell type names are already in the data
unique_celltypes = adata.obs['cell_type'].unique()
cell_names_mapping = {celltype: celltype for _, celltype in enumerate(unique_celltypes)}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping

config.cell_names = neighbor_normalized_df.columns
config.gene_names = top_genes

config.system_prompt = prompt.CP_celltype_geneorder(one_shot_df, config)
if config.with_negatives:
    # get top 3 genes for each row in one_shot_df
    neg_top_3_genes = one_shot_df[config.gene_names].apply(lambda x: x.nlargest(3).index.tolist(), axis=1)
    config.neg_top_3_genes = neg_top_3_genes


In [ ]:
print(config.data_name)
print(config.replicate)
print(config.gpt_model)

In [ ]:
generate_json_end2end(neighbor_normalized_df, config, prompt_func=prompt.finetune_user_celltype_geneorder, n_rows=1, batch_size=3000, max_completion_tokens=128, df_extra=neighbor_normalized_df_genes)

In [ ]:
# submit_end2end.py
# nohup python -u -m src.submit_end2end configs/config_BZ9_zeroshot.yaml > BZ9_zeroshot.out 2>&1 &
import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_finetunePro_merfish.yaml {config.data_name} {config.replicate} > outs/{config.data_name}_finetunePro_25{config.replicate}_{config.prototype_p}.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 2
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ").replace("‘", "'").replace("’", "'")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")


In [ ]:
gpt_results_df

In [ ]:
gpt_results_df = gpt_results_df[[0]]
gpt_results_df.columns = ['finetunePro_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
gpt_results_df.value_counts()

In [ ]:
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "Outputs: 'PV']", "finetunePro_gpt4o_mini"] = "PV"

In [ ]:
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "FX", "finetunePro_gpt4o_mini"] = "fx"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "MP A", "finetunePro_gpt4o_mini"] = "MPA"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == " BST", "finetunePro_gpt4o_mini"] = "BST"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "MPn", "finetunePro_gpt4o_mini"] = "MPN"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "'PV''", "finetunePro_gpt4o_mini"] = "PV"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "mpn", "finetunePro_gpt4o_mini"] = "MPN"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "Portal Hypothalamus (PVH)", "finetunePro_gpt4o_mini"] = "PVH"







In [ ]:
# if the number of the cell type is less than 4, set it to unknown
for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<3].index:
    gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = "unknown"

## plot and save

In [ ]:
# import matplotlib.pyplot as plt
# from matplotlib.colors import TwoSlopeNorm

# for gene in ['Ebf3', 'Fn1', 'Calcr', 'Klf4', 'Adcyap1']:
#     plt.figure()
#     norm = TwoSlopeNorm(vmin=neighbor_normalized_df_genes[gene].min(), vcenter=neighbor_normalized_df_genes[gene].max()/2, vmax=neighbor_normalized_df_genes[gene].max())
#     plt.scatter(adata.obsm['spatial'][:,0], 
#                 adata.obsm['spatial'][:,1], 
#                 #c=adata[:, gene].X.toarray().flatten(), 
#                 c=neighbor_normalized_df_genes[gene],
#                 cmap='viridis',
#                 s=4,
#                 norm=norm)
#     plt.colorbar(label=gene)
#     plt.title(f'{gene} expression')
#     plt.show()


In [ ]:
# gpt_results_df = pd.read_csv(f"./finetune_results/MERFISH25/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv", index_col=0)
# gpt_results_df.index = gpt_results_df.index.astype(str)

In [ ]:
# adata.obs = gpt_results_df

In [ ]:
adata.obs = adata.obs.join(gpt_results_df)
# replace the NA in adata.obs['finetune_gpt4o'] with "unknown"
adata.obs['finetunePro_gpt4o_mini'] = adata.obs['finetunePro_gpt4o_mini'].fillna("unknown")

sc.pl.scatter(adata, x="x", y="y", color="finetunePro_gpt4o_mini", title =  f"finetunePro_gpt4o_mini")
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini']))
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini']))

In [ ]:
# gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
# print(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
# refine the niche
config.r_factor = 1
refined_niche = relabel_cells(adj_matrix.toarray(), gpt_results_df['finetunePro_gpt4o_mini'])
adata.obs['finetunePro_gpt4o_mini_refined'] = refined_niche


In [ ]:
sc.pl.scatter(adata, x="x", y="y", color="finetunePro_gpt4o_mini_refined", title =  f"finetunePro_gpt4o_mini_refined")

print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_refined']))
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_refined']))

In [ ]:
save_folder = "./finetune_results/MERFISH25"
adata.obs.to_csv(f"{save_folder}/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
# Run this in second stage
save_folder = "./finetune_results/MERFISH25"
adata.obs = pd.read_csv(f"{save_folder}/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv", index_col=0)
adata.obs.index = adata.obs.index.astype(str)

## high confidence cells

In [ ]:
# neighbor_combined_df = neighbor_normalized_df.join(neighbor_normalized_df_genes).copy()

In [ ]:
# # run kmeans multiple times and see if the conserved groups are stable
# kmeans_results = pd.DataFrame()
# random_times = 100
# for random_state in range(1,random_times+1):
#     km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=np.random.randint(1,1000))
#     # clusters = km.fit_predict(pd.DataFrame(adata.X))
#     clusters = km.fit_predict(neighbor_combined_df)
#     kmeans_results[f'kmeans_both_{random_state}'] = clusters.astype(str)

# conserved_groups = find_conserved_groups(kmeans_results, min_methods=int(random_times*1), min_group_size=30)

# # Print results
# for group_num, (cells, info) in enumerate(conserved_groups.items(), 1):
#     print(f"\nConserved Group {group_num}:")
#     print(f"Number of cells: {len(cells)}")

In [ ]:


# # Create group labels DataFrame
# group_labels_df = create_group_labels_df(conserved_groups, 
#                                        all_cell_indices=neighbor_combined_df.index)

# # Add to adata.obs if needed
# adata.obs['conserved_groups'] = group_labels_df['conserved_label']

In [ ]:
# adata.uns.pop('conserved_groups_colors')


In [ ]:
# sc.pl.scatter(adata, x="x", y="y", color="conserved_groups", title =  f"conserved_groups")

In [ ]:
# # filter out the unknown
# conserved_obs = adata.obs.loc[adata.obs['conserved_groups'] != "unassigned"].copy()
# # Get cross tabulation of the two groupings
# # the order of the columns matters
# cross_tab = pd.crosstab(conserved_obs['conserved_groups'],
#                         conserved_obs['finetunePro_gpt4o_mini_refined']
#                        )

# # Find dominant conserved group for each refined group
# dominant_groups = {}
# conserved_cells = {}

# for refined_group in cross_tab.index:
#     # Get the most common conserved group(s) for this refined group
#     max_count = cross_tab.loc[refined_group].max()
#     dominant_conserved = cross_tab.loc[refined_group][cross_tab.loc[refined_group] == max_count].index.tolist()
    
#     # Store the mapping
#     dominant_groups[refined_group] = dominant_conserved
    
#     # Get cell indices where both labels match
#     mask = (conserved_obs['finetunePro_gpt4o_mini_refined'] == refined_group) & \
#            (conserved_obs['conserved_groups'].isin(dominant_conserved))
#     conserved_cells[refined_group] = conserved_obs.index[mask].tolist()
    


In [ ]:
# # Map each key to the first value in its list
# dominant_groups = {k: v[0] for k, v in dominant_groups.items()}
# # Map the values to conserved_obs['conserved_groups']
# conserved_obs['conserved_groups'] = conserved_obs['conserved_groups'].map(dominant_groups)
# conserved_obs['conserved_groups']

In [ ]:
# adata.obs['post_domain'] = adata.obs['conserved_groups'].copy()
# adata.obs['post_domain'] = adata.obs['post_domain'].astype(str)
# adata.obs.loc[conserved_obs.index, 'post_domain'] = conserved_obs['conserved_groups']

In [ ]:
# sc.pl.scatter(adata, x="x", y="y", color="post_domain", title =  f"post_domain")

### get confident cells
get confident cells based on the r


In [ ]:
label_key = 'finetunePro_gpt4o_mini'  # IMPORTANT!!!!! before refinement
high_confidence_mask = find_high_confidence_cells(
    adata,
    label_key=label_key,
    k=min(30, int((np.mean(n_neighbors))/2)),  # Consider top 20/ half of the mean nearest neighbors  # IMPORTANT!!!!!!  others are min(20) only merfish29 is min(30)
    distance_threshold=config.r
)

In [ ]:
adata.obs['confident_cells'] = adata.obs['finetunePro_gpt4o_mini'].copy()
adata.obs['confident_cells'] = adata.obs['confident_cells'].astype(str)
adata.obs.loc[~high_confidence_mask, 'confident_cells'] = 'unconfident'
sc.pl.scatter(adata, x="x", y="y", color="confident_cells", title =  f"confident_cells")


In [ ]:


# Get unique values excluding 'unconfident'
confident_unique = set(adata.obs.loc[high_confidence_mask, 'confident_cells'].unique()) - {'unconfident'}
unique_layers = set(adata.obs[label_key].unique()) - {'unknown'}
# Check if any elements are missing
missing_elements = unique_layers - confident_unique
if len(missing_elements) > 0:
    print(f"Missing elements in confident cells: {missing_elements}")
    print("Adding missing elements to high_confidence_mask")
    missing_mask = adata.obs[label_key].isin(missing_elements)
    high_confidence_mask = high_confidence_mask | missing_mask
else:
    print("No missing elements in confident cells") 
    

    


### prototype for second stage

In [ ]:
# # merge all conserved cells index together
# conserved_cells_index = [item for sublist in conserved_cells.values() for item in sublist]

# conserved_normalized_df = neighbor_normalized_df.loc[conserved_cells_index]
# conserved_normalized_df_genes = neighbor_normalized_df_genes.loc[conserved_cells_index]

# un_conserved_cells_index = adata.obs_names[~adata.obs_names.isin(conserved_cells_index)]
# un_conserved_normalized_df = neighbor_normalized_df.loc[un_conserved_cells_index]
# un_conserved_normalized_df_genes = neighbor_normalized_df_genes.loc[un_conserved_cells_index]   

# print(f"find {len(conserved_cells_index)} conserved cells")
# print(f"find {len(un_conserved_cells_index)} un-conserved cells")

In [ ]:
# merge all conserved cells index together

conserved_normalized_df = neighbor_normalized_df.loc[high_confidence_mask]
conserved_normalized_df_genes = neighbor_normalized_df_genes.loc[high_confidence_mask]

un_conserved_normalized_df = neighbor_normalized_df.loc[~high_confidence_mask]
un_conserved_normalized_df_genes = neighbor_normalized_df_genes.loc[~high_confidence_mask]   

print(f"find {len(conserved_normalized_df)} conserved cells")
print(f"find {len(un_conserved_normalized_df)} un-conserved cells")

In [ ]:
print(f"conserved_json/{config.data_name}_{config.model_type}/" )

In [ ]:
# IMPORTANT!!!!!
# prototype is from conserved cells

# calculate prototype
conserved_neighbor_df = conserved_normalized_df.join(conserved_normalized_df_genes).copy()
one_shot_df = pd.concat([adata.obs[label_key].loc[conserved_neighbor_df.index], conserved_neighbor_df], axis=1).groupby(label_key, observed=False).mean()
# remove unknown
if 'unknown' in one_shot_df.index:
    one_shot_df = one_shot_df.drop(index='unknown')

config.system_prompt = prompt.CP_celltype_geneorder(one_shot_df, config)
if config.with_negatives:
    # get top 3 genes for each row in one_shot_df
    neg_top_3_genes = one_shot_df[config.gene_names].apply(lambda x: x.nlargest(3).index.tolist(), axis=1)
    config.neg_top_3_genes = neg_top_3_genes

In [ ]:
print(config.system_prompt)

### oneshot with conserved

In [ ]:
print(config.folder_path)
print(config.gpt_model)
config.replicate = "_rep4R100_unconserved"  # "_rep2R100_unconserved" is for one shot, _rep2R100_finetune is for finetune


In [ ]:
# IMPORTANT!!!!!
# json is from un-conserved cells
generate_json_end2end(un_conserved_normalized_df, config, prompt_func=prompt.finetune_user_celltype_geneorder, max_completion_tokens=128, batch_size=3000, df_extra=un_conserved_normalized_df_genes)


In [ ]:
# submit_end2end.py

import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_finetunePro_merfish.yaml {config.data_name} {config.replicate} > outs/{config.data_name}_finetunePro_25{config.replicate}_{config.prototype_p}.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 2

for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['finetunePro_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)
gpt_results_df.index = gpt_results_df.index.str.replace("id_", "")


In [ ]:
adata.obs['finetunePro_gpt4o_mini_twostage'] = adata.obs[label_key].copy()
adata.obs['finetunePro_gpt4o_mini_twostage'] = adata.obs['finetunePro_gpt4o_mini_twostage'].astype(str)

In [ ]:
gpt_results_df.finetunePro_gpt4o_mini.value_counts()

In [ ]:
# fill nan with unknown
gpt_results_df.fillna("unknown", inplace=True)


In [ ]:
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "Outputs: 'BST'", "finetunePro_gpt4o_mini"] = "BST"

In [ ]:
# if the number of the cell type is less than 4, set it to unknown
for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<4].index:
    gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = "unknown"

In [ ]:
adata.obs.loc[gpt_results_df.index, 'finetunePro_gpt4o_mini_twostage'] = gpt_results_df.finetunePro_gpt4o_mini

In [ ]:
sc.pl.scatter(adata, x="x", y="y", color="finetunePro_gpt4o_mini_twostage", title =  f"finetunePro_gpt4o_mini_twostage")

In [ ]:
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_twostage']))

In [ ]:
# refine the niche
adj_matrix, _ = sparse_adjacency(pos_data.loc[neighbor_normalized_df.index], threshold=r)

refined_niche = relabel_cells(adj_matrix.toarray(), adata.obs['finetunePro_gpt4o_mini_twostage'])

adata.obs['finetunePro_gpt4o_mini_twostage_refined'] = refined_niche


In [ ]:
sc.pl.scatter(adata, x="x", y="y", color="finetunePro_gpt4o_mini_twostage_refined", title =  f"finetunePro_gpt4o_mini_twostage_refined")

In [ ]:
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_twostage_refined']))

In [ ]:
save_folder = './twostage_results'
adata.obs.to_csv(f"{save_folder}/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")



In [ ]:

# adata.obs= pd.read_csv(f"./twostage_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

### finetune with conserved

In [ ]:
config.replicate = "_rep2R100_conserved"  # IMPORTANT!!!!! change conserved/ unconserved


In [ ]:
# generate json for finetune
output_folder = f"conserved_json/{config.data_name}_{config.model_type}/"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

train_output_file = f"{output_folder}{config.data_name}_conserved_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
print(f"Generating json for training into {train_output_file}")
with open(train_output_file, 'w') as f:
    for i in range(conserved_normalized_df.shape[0]):
        system_p = config.system_prompt
        user_p = prompt.finetune_user_celltype_geneorder(conserved_normalized_df, conserved_normalized_df_genes, i, config)
        assistant_p = prompt.finetune_assistant(conserved_normalized_df, i, adata.obs['finetunePro_gpt4o_mini'])
        
        row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
        json_str = json.dumps(row_data)
        f.write(json_str + '\n')  

In [ ]:
print(system_p + user_p + "\n" + assistant_p)

In [ ]:
train_file = client.files.create(
  file=open(train_output_file, "rb"),
  purpose="fine-tune"
)


finetune_job = client.fine_tuning.jobs.create(
  training_file=train_file.id,
  model="ft:gpt-4o-mini-2024-07-18:personal:finetunepro-merfish250-3-100-newgenes:AWXjqn26",
  suffix=f"{config.model_type}_{config.r}_secondstage_rep2"  # default is personal
)


## gather replicate results

In [ ]:
config = load_config("configs/config_finetunePro_merfish.yaml")

In [ ]:
print(config.data_name)
print(config.model_type)
print(config.prototype_p)

In [ ]:
save_folder = "./finetune_results/MERFISH25"

In [ ]:
## gather replicate results

if len(neighbor_normalized_df) > 3000:
    n_batch = 2  
else:
    n_batch = 1
config.r_factor = 1
adj_matrix, _ = sparse_adjacency(pos_data.loc[neighbor_normalized_df.index], threshold=r*config.r_factor)


for replicate in ['_rep1R100', '_rep2R100', '_rep3R100']:
    config.replicate = replicate
    # 初始化空列表以保存custom_id和content
    gpt_results_df = pd.DataFrame()
    for i in range(1,n_batch+1):
        save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
        output_file_name = f"{config.output_path}/{save_name}"
        # 打开文件并逐行读取
        with open(output_file_name, 'r', encoding='utf-8') as file:
            for line in file:
                try:
                    # 解析每一行的json字符串
                    json_data = json.loads(line.strip())
                    
                    # 提取custom_id和content信息
                    custom_id = json_data['custom_id']
                    content = json_data['response']['body']['choices'][0]['message']['content']
                    content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                    content = content.replace("Layer ", "Layer")
                    # extract outputs
                    extract_dict = extract_output_microenvironments(content)
                    # extract_dict = extract_last_braces(content)

                    # 将提取到的信息添加到数据框中
                    gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                        
                except json.JSONDecodeError:
                    print(f"无法解析JSON字符串: {line}")
    gpt_results_df.columns = ['finetunePro_gpt4o_mini']
    gpt_results_df.index = gpt_results_df.index.astype(str)
    gpt_results_df.index = gpt_results_df.index.str.replace("id_", "")

    # Get the list of keywords from domain_mapping
    keywords = list(domain_mapping.values())

    # Create a regex pattern to capture any keyword possibly surrounded by other text
    pattern = r'.*(' + '|'.join(map(re.escape, keywords)) + r')[\*\.\s]*.*'

    # Replace the entire string with the captured keyword only if it could be not unknown
    for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<3].index:
        gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"].str.replace(pattern, r'\1', regex=True)

    # if the number of the cell type is less than 3, set it to unknown
    for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<3].index:
        gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = "unknown"
    
    if 'unknown' in gpt_results_df.finetunePro_gpt4o_mini.value_counts().index:
        print(f"unknown: {gpt_results_df.value_counts()['unknown']}")

    # merge results to obs
    val_adata = adata.copy()
    val_adata.obs = val_adata.obs.join(gpt_results_df)
    val_adata.obs['finetunePro_gpt4o_mini'] = val_adata.obs['finetunePro_gpt4o_mini'].fillna("unknown")

    # refine the niche
    refined_niche = relabel_cells(adj_matrix.toarray(), val_adata.obs['finetunePro_gpt4o_mini'])

    val_adata.obs['finetunePro_gpt4o_mini_refined'] = refined_niche
    print(f"ARI: {adjusted_rand_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini'])}")
    print(f"NMI: {normalized_mutual_info_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini'])}")
    print(f"ARI refined: {adjusted_rand_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini_refined'])}")
    print(f"NMI refined: {normalized_mutual_info_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini_refined'])}")

    val_adata.obs.to_csv(f"{save_folder}/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")



.*(MPA|MPN|BST|fx|PVH|PVT|V3|PV)[\*\.\s]*.*